## Neighbourhood simulation
Where does a sound-driven signal actually go in the real wiring, when the path is not picked in advance?

---

The idea: 03 only ran charge along a hand-picked path, so reaching the end was close to guaranteed
- instead, grow a bounded neighbourhood outward from the JO-B neurons, one hop at a time
- read only the edges touching neurons already reached, never the whole connectome
- then run LIF on that neighbourhood and see which motor neurons actually fire

Why do this at all?
- It is already known, from behaviour, that a fly hears something and moves its legs. This notebook does not try to prove that.
- What it checks is whether the real wiring in this dataset can carry that signal through when it is modelled as simple charge and leak, and how long it takes. Behaviour says what happens at the outside, the simulation checks the mechanism underneath.

Why not reuse the hand-picked path?
- 03 and 04 followed a path chosen in advance, so reaching the end was close to guaranteed, and the test could not surprise anyone.
- Here the signal starts at the hearing neurons and is given all the real nearby connections.
- Where it ends up is then a result, not a setup. It could reach the expected jump/leg motor neurons, unexpected ones, or die out.

Why a neighbourhood and not the whole connectome?
- The full data is ~166k neurons and 151M connections, most of it irrelevant to a sound pulse.
- Growing outward a few hops from the hearing neurons keeps it small enough to run and to read.

Baseline assumed: a healthy fly in normal condition, hears a sound, nothing else competing (no smell, vision or internal state).

Known limits, stated up front:
- the edge weights are unsigned synapse counts, so the model can only excite, never inhibit
- inputs from outside the neighbourhood are missing
- the weight cutoff and hop limit are choices, so they get tested in a sensitivity check

So a result here is a claim about the strong nearby wiring only, not a claim about what the fly does in general.

### Init

In [8]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc
import duckdb
import networkx as nx

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

df = pd.read_feather(neurons_file)
traced = df[df['status'] == 'Traced']
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

# lazy handle on the big file, nothing is loaded until a filtered read asks for it
dataset = ds.dataset(connections_file, format="feather")

#### Pulling out the neighbourhood around the hearing neurons

In [ ]:
jo_test_id = 42619  # JO-B1_a, the neuron hand-traced in 04

# get outgoing connections of specified neurons
def query_edges(neuron_ids, min_weight=3):
    if not isinstance(neuron_ids, (list, set, tuple)):
        neuron_ids = [neuron_ids]
    expr = pc.field("body_pre").isin(neuron_ids) & (pc.field("weight") >= min_weight)
    return dataset.to_table(filter=expr).to_pandas()

edges = query_edges(jo_test_id)
print(edges.shape)
print(edges.sort_values('weight', ascending=False).head())

(60, 3)
   body_pre  body_post  weight
0     42619      15233      46
1     42619     525475      45
2     42619      13014      42
3     42619      11378      42
4     42619      12066      39


> the top row matches what was seen in `04`

next, check if the function drops anything it shouldn't:

In [10]:
all_out = query_edges(jo_test_id, min_weight=1)
print(all_out.shape)
print(all_out)

# code pulled from 04
filter_expr = (pc.field("body_pre") == jo_test_id) | (pc.field("body_post") == jo_test_id)
jo_neighbours = dataset.to_table(filter=filter_expr).to_pandas()

# in 04, jo_neighbours had both directions, so keep only the outgoing rows
print((jo_neighbours['body_pre'] == jo_test_id).sum())


(229, 3)
     body_pre  body_post  weight
0       42619      15233      46
1       42619     525475      45
2       42619      13014      42
3       42619      11378      42
4       42619      12066      39
..        ...        ...     ...
224     42619      13306       1
225     42619      17278       1
226     42619      10105       1
227     42619      11811       1
228     42619      11950       1

[229 rows x 3 columns]
229


> confirms the function is correct: it returns the same outgoing edges as 04, and the cutoff behaves as expected.

stage 1's first piece is done, the next piece is the loop around it. after one hop, decide which of those neurons to query next. 

the simplest option would be to query all targets from hop 1. 

but to prevent loops, add all the body_post id's to a "seen" set. and when carrying out a hop, check if it's in the seen set or not. if not, then it is a next candidate for a hop and will be seen, then repeat.

In [ ]:
def grow_neighbourhood(start_ids, n_hops, min_weight=3):
    if not isinstance(start_ids, (list, set, tuple)):
        start_ids = [start_ids]

    seen = set(start_ids)       # every neuron reached so far, starting ones included
    to_check = set(start_ids)   # neurons whose outgoing connections we look up this hop
    all_edges = []

    for hop in range(1, n_hops + 1):
        hop_edges = query_edges(to_check, min_weight=min_weight)
        all_edges.append(hop_edges)

        to_check = set(hop_edges['body_post']) - seen   # targets not seen before
        # seen |= to_check
        seen.update(to_check)

        # prints each hop statistics
        print(f"hop {hop}: {len(hop_edges)} edges, {len(to_check)} new neurons, {len(seen)} total")
        if not to_check:
            break

    return pd.concat(all_edges, ignore_index=True), seen

> note: the seen set only decides which neurons get queried again. every edge a query returns is kept, it's just the same neurons won't be constantly re queryed for edges.

trying the function:

In [12]:
edges_1, seen_1 = grow_neighbourhood(jo_test_id, n_hops=1)

edges_2, seen_2 = grow_neighbourhood(jo_test_id, n_hops=2)

hop 1: 60 edges, 60 new neurons, 61 total
hop 1: 60 edges, 60 new neurons, 61 total
hop 2: 7081 edges, 4400 new neurons, 4461 total


> when filtering for min_weight >= 3, the same results are seen for one hop as previously in this notebook `(60, 3)`.

the hand traced path was multiple hops: cb_sensory -> cb_intrinsic -> descending_neuron -> vnc_intrinsic -> vnc_motor

so four hops would be made. check first how big the graph would get:

In [13]:
edges_4, seen_4 = grow_neighbourhood(jo_test_id, n_hops=4)

hop 1: 60 edges, 60 new neurons, 61 total
hop 2: 7081 edges, 4400 new neurons, 4461 total
hop 3: 539731 edges, 342343 new neurons, 346804 total
hop 4: 8985411 edges, 5219030 new neurons, 5565834 total


> many edges -> try and find a higher weight cut off. however try not to cutoff connections based on the observed chain found in `04`, or else the expected route survives, and the answer is predetermined before even running.

what should be observed is how that charge flows to other neurons and not just those in the chain.

treat the chain as a check, not a target. pick the cutoff for size, then see whether the chain survives, and report it either way

In [15]:
edges['pre_type'] = edges['body_pre'].map(id_to_superclass)
edges['post_type'] = edges['body_post'].map(id_to_superclass)

core = ['cb_sensory', 'cb_intrinsic', 'descending_neuron', 'vnc_intrinsic', 'vnc_motor']
core_edges = edges[edges['pre_type'].isin(core) & edges['post_type'].isin(core)]

print(core_edges.groupby(['pre_type', 'post_type'])['weight'].describe(percentiles=[.5, .9, .99]))

# na counts
print(edges['post_type'].value_counts(dropna=False))

                              count       mean        std  min   50%   90%  \
pre_type   post_type                                                         
cb_sensory cb_intrinsic        28.0  16.571429  14.915636  3.0  10.0  42.0   
           descending_neuron    1.0   5.000000        NaN  5.0   5.0   5.0   

                                99%   max  
pre_type   post_type                       
cb_sensory cb_intrinsic       45.73  46.0  
           descending_neuron   5.00   5.0  
post_type
NaN                  31
cb_intrinsic         28
descending_neuron     1
Name: count, dtype: int64


> note on percentiles:
> - 50% (the median) is the weight of the edge in the middle. Half the edges are weaker than this, and half are stronger. It's the "typical" connection.
> - 90% is the weight that 90 of the 100 edges don't exceed. Only the strongest 10 edges are above it.
> - 99% is the same idea at the extreme end, where only the single strongest edge is above it.
>
> also the reason why there are 28 + 1 in count is because there are 29 nodes that are not annotated, as found before

however this was not useful, as hop 4 would be better to look at to see what weights should be filtered:

In [16]:
edges_4['pre_type'] = edges_4['body_pre'].map(id_to_superclass)
edges_4['post_type'] = edges_4['body_post'].map(id_to_superclass)

core_edges = edges_4[edges_4['pre_type'].isin(core) & edges_4['post_type'].isin(core)]
print(core_edges.groupby(['pre_type', 'post_type'])['weight'].describe(percentiles=[.5, .9, .99]))

                                         count       mean        std  min  \
pre_type          post_type                                                 
cb_intrinsic      cb_intrinsic       1378853.0  12.573975  22.341762  3.0   
                  cb_sensory           35897.0   6.870184   7.118198  3.0   
                  descending_neuron   118929.0  18.269858  33.089189  3.0   
                  vnc_intrinsic           32.0   9.062500   7.678783  3.0   
                  vnc_motor                6.0  12.833333  11.788412  3.0   
cb_sensory        cb_intrinsic         32234.0  10.449929  12.739957  3.0   
                  cb_sensory            5402.0   5.457793   5.024857  3.0   
                  descending_neuron     8038.0  10.991789  15.181894  3.0   
descending_neuron cb_intrinsic         57846.0  11.740241  20.029060  3.0   
                  cb_sensory             877.0   8.481186  10.363867  3.0   
                  descending_neuron    30438.0  17.213089  31.594852  3.0   

> a lot of information to look at. but looking in particular at steps in the chain established, the typical connection is weak. the median is 6 -> 10.
>
> a cut off of 10 would keep roughly half the edges. to keep the strongest tenth, the cutoff would be somewhere between 22 and 60 depending on the step.

remembering from `04`, the weakest link was weight 12 so any cutoff above 12 would remove it. but also try not to tailor this cutoff to the established chain. below is experimenting the size of the simulation given different cutoffs:

In [ ]:
for cutoff in [10, 20, 40]:
    print("min_weight", cutoff)
    grow_neighbourhood(jo_test_id, n_hops=4, min_weight=cutoff)

min_weight 10
hop 1: 15 edges, 15 new neurons, 16 total
hop 2: 815 edges, 338 new neurons, 354 total
hop 3: 20718 edges, 6495 new neurons, 6849 total
hop 4: 350953 edges, 49504 new neurons, 56353 total
min_weight 20
hop 1: 9 edges, 9 new neurons, 10 total
hop 2: 237 edges, 149 new neurons, 159 total
hop 3: 4177 edges, 1646 new neurons, 1805 total
hop 4: 59766 edges, 14794 new neurons, 16599 total
min_weight 40
hop 1: 4 edges, 4 new neurons, 5 total
hop 2: 36 edges, 31 new neurons, 36 total
hop 3: 440 edges, 287 new neurons, 323 total
hop 4: 4354 edges, 2263 new neurons, 2586 total


> cutoff 10 is a reasonable pick.
>
> the 49,504 neurons found at hop 4 have no outgoing edges in the table (the last hop was never looked up),
>
> so in the simulation they can receive charge but never pass it on. Feedback from them to earlier neurons is not modelled.

#### LIF Model

